In [1]:
import duckdb

In [8]:
cd /Users/abi/Project5/edtech_churn_engine

/Users/abi/Project5/edtech_churn_engine


In [9]:

# 1. Establish persistent DuckDB database connection
con = duckdb.connect('edtech_churn_engine.duckdb')

# 2. Ingest raw CSV files into DuckDB tables
con.execute("""
CREATE TABLE IF NOT EXISTS raw_student_info AS 
SELECT * FROM read_csv_auto('data/studentInfo.csv', header=True);

CREATE TABLE IF NOT EXISTS raw_student_registration AS 
SELECT * FROM read_csv_auto('data/studentRegistration.csv', header=True);

CREATE TABLE IF NOT EXISTS raw_student_assessment AS 
SELECT * FROM read_csv_auto('data/studentAssessment.csv', header=True);

CREATE TABLE IF NOT EXISTS raw_assessments AS 
SELECT * FROM read_csv_auto('data/assessments.csv', header=True);

CREATE TABLE IF NOT EXISTS raw_vle AS 
SELECT * FROM read_csv_auto('data/vle.csv', header=True);

CREATE TABLE IF NOT EXISTS raw_student_vle AS 
SELECT * FROM read_csv_auto('data/studentVle.csv', header=True);
""")

print("Raw ingestion complete! Total VLE Clickstream Logs:", 
      con.execute("SELECT COUNT(*) FROM raw_student_vle").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw ingestion complete! Total VLE Clickstream Logs: 10655280


In [11]:
# Create Staging Views in DuckDB
con.execute("""
-- 1. Staging View: Student Registrations & Unregistrations
CREATE OR REPLACE VIEW stg_student_registrations AS
SELECT
    TRIM(code_module) AS module_code,
    TRIM(code_presentation) AS presentation_code,
    CAST(id_student AS BIGINT) AS student_id,
    
    -- Negative dates indicate registration prior to official course start date
    CAST(date_registration AS INTEGER) AS registration_day_offset,
    CAST(date_unregistration AS INTEGER) AS unregistration_day_offset,
    
    -- Derived Business Logic Flags
    CASE 
        WHEN date_unregistration IS NOT NULL THEN TRUE 
        ELSE FALSE 
    END AS has_withdrawn,
    
    CASE 
        WHEN date_unregistration IS NOT NULL AND date_unregistration <= 14 THEN TRUE 
        ELSE FALSE 
    END AS is_early_withdrawal_14d
FROM raw_student_registration
WHERE id_student IS NOT NULL;


-- 2. Staging View: Assessment Performance & Module Weights
CREATE OR REPLACE VIEW stg_assessment_results AS
SELECT
    a.code_module AS module_code,
    a.code_presentation AS presentation_code,
    CAST(a.id_assessment AS BIGINT) AS assessment_id,
    TRIM(a.assessment_type) AS assessment_type,
    CAST(a.date AS INTEGER) AS assessment_due_day,
    CAST(a.weight AS DECIMAL(5,2)) AS assessment_weight,
    
    CAST(sa.id_student AS BIGINT) AS student_id,
    CAST(sa.date_submitted AS INTEGER) AS date_submitted,
    
    -- Handle missing assessment scores
    COALESCE(CAST(sa.score AS DECIMAL(5,2)), 0.00) AS raw_score,
    
    -- Submission timeliness indicator (Positive = Late Submission)
    (CAST(sa.date_submitted AS INTEGER) - CAST(a.date AS INTEGER)) AS submission_delay_days,
    
    -- Pass/Fail Marker (Standard threshold = 40)
    CASE 
        WHEN COALESCE(CAST(sa.score AS DECIMAL(5,2)), 0.00) >= 40.00 THEN TRUE 
        ELSE FALSE 
    END AS is_passed
FROM raw_assessments a
INNER JOIN raw_student_assessment sa
    ON a.id_assessment = sa.id_assessment;


-- 3. Staging View: VLE Clickstream Logs (10M+ Event Records)
CREATE OR REPLACE VIEW stg_vle_interactions AS
SELECT
    vle.code_module AS module_code,
    vle.code_presentation AS presentation_code,
    CAST(sv.id_student AS BIGINT) AS student_id,
    CAST(sv.id_site AS BIGINT) AS vle_site_id,
    TRIM(vle.activity_type) AS activity_type,
    
    CAST(sv.date AS INTEGER) AS interaction_day_offset,
    CAST(sv.sum_click AS INTEGER) AS daily_click_count,
    
    -- Classify learning phase based on course launch day (Day 0)
    CASE 
        WHEN sv.date < 0 THEN 'Pre-Course Preparation'
        WHEN sv.date BETWEEN 0 AND 14 THEN 'Week 1-2 Critical Window'
        ELSE 'Active Term'
    END AS learning_phase
FROM raw_student_vle sv
LEFT JOIN raw_vle vle
    ON sv.id_site = vle.id_site
WHERE sv.sum_click > 0;
""")

print("Staging views created successfully!")

Staging views created successfully!


In [ ]:
# Diagnostic Query A: Summary of Student Registrations vs Early 14-Day Withdrawals
df_reg_audit = con.execute("""
    SELECT 
        module_code,
        presentation_code,
        COUNT(DISTINCT student_id) AS total_enrolled,
        SUM(CASE WHEN has_withdrawn THEN 1 ELSE 0 END) AS total_withdrawals,
        SUM(CASE WHEN is_early_withdrawal_14d THEN 1 ELSE 0 END) AS early_14d_withdrawals,
        ROUND(SUM(CASE WHEN is_early_withdrawal_14d THEN 1 ELSE 0 END)::DOUBLE / COUNT(DISTINCT student_id) * 100, 2) AS early_withdrawal_pct
    FROM stg_student_registrations
    GROUP BY module_code, presentation_code
    ORDER BY total_enrolled DESC
    LIMIT 5;
""").df()

print("--- Enrollment & Early Withdrawal Audit ---")
print(df_reg_audit)

# Diagnostic Query B: Clickstream Activity Volume by Learning Phase
df_vle_audit = con.execute("""
    SELECT 
        learning_phase,
        COUNT(DISTINCT student_id) AS active_students,
        SUM(daily_click_count) AS total_clicks,
        ROUND(AVG(daily_click_count), 2) AS avg_clicks_per_interaction
    FROM stg_vle_interactions
    GROUP BY learning_phase
    ORDER BY total_clicks DESC;
""").df()

print("\n--- Clickstream Log Volume Audit ---")
print(df_vle_audit)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Enrollment & Early Withdrawal Audit ---
  module_code presentation_code  total_enrolled  total_withdrawals  \
0         CCC             2014J            2498             1049.0   
1         FFF             2014J            2365              831.0   
2         BBB             2014J            2292              736.0   
3         FFF             2013J            2283              677.0   
4         BBB             2013J            2237              647.0   

   early_14d_withdrawals  early_withdrawal_pct  
0                  414.0                 16.57  
1                  477.0                 20.17  
2                  482.0                 21.03  
3                  285.0                 12.48  
4                  383.0                 17.12  

--- Clickstream Log Volume Audit ---
             learning_phase  active_students  total_clicks  \
0               Active Term            24600    33409797.0   
1  Week 1-2 Critical Window            23913     4047355.0   
2    Pre-Course P

: 